# 04 — Hedge-Ratio Comparison

**Purpose:** Select hedge methodology per pair by comparing all candidate estimators on residual quality, stability, turnover, and OOS behavior (mandate notebook 04).

**Research questions:**
1. Which method minimizes OOS residual volatility and directional leakage?
2. How stable is each method's ratio (turnover cost of tracking it)?
3. How much directional exposure does integer-contract rounding leave?
4. How sensitive is each method to lookback choice?

**Data used:** minute data for all 7 pairs (**blocked**, L-001).


In [ ]:
import sys
sys.path.insert(0, "../src")
import numpy as np
import pandas as pd
import yaml

RESEARCH_CONFIG = yaml.safe_load(open("../config/research_config.yaml"))
SEED = RESEARCH_CONFIG["meta"]["random_seed"]
np.random.seed(SEED)
print(f"config loaded | global seed = {SEED}")


## Methodology

Index methods: contract-notional, rolling OLS, robust (Huber), EW, dollar-vol, Kalman. Treasury methods: 1:1 deliberately-weak baseline, dollar-vol, rolling OLS, robust, Kalman, approximate DV01. All via `spread_research.hedge_ratios` (every estimator look-ahead-safe, `shifted=True`). Evaluation on a train/validation split (`research_config.validation`): OOS residual σ, net-notional drift, `hedge_stability` turnover, `position_sizing.size_spread` rounding error, lookback-grid sensitivity via `sensitivity.run_grid` + `plateau_score`. **Selection rule (D-005 spirit): most-profitable-in-backtest is NOT the criterion — residual quality and stability are.** L-007 (hedge-noise inflation of half-life) is quantified here on real data.

In [ ]:
from pathlib import Path

DATA_DIR = Path("../data/processed")
DATA_AVAILABLE = any(DATA_DIR.glob("*_minute.*")) if DATA_DIR.exists() else False
if not DATA_AVAILABLE:
    print("BLOCKED-ON-DATA: no futures market data in this environment (see "
          "reports/00_repository_audit.md, issue L-001).\n"
          "Run this notebook inside QuantConnect Research, or drop licensed data\n"
          "into data/processed/ in the canonical schema (src/spread_research/data_loader.py).")


In [ ]:
if DATA_AVAILABLE:
    from spread_research import hedge_ratios as hr
    from spread_research.pair_builder import align_pair, build_residual
    from spread_research.sensitivity import run_grid, plateau_score
    print("wire per-pair comparison here per methodology above")

## Results

**BLOCKED-ON-DATA** — this section intentionally contains no results. No synthetic or fabricated market findings are presented as evidence (CLAUDE.md gate 3). It will be populated when the notebook runs against real data.

## Limitations

Kalman delta and EW halflife are themselves tuned parameters — they enter the multiple-testing count.

## Decision

Hedge method per pair selected and logged (research_decisions.md) before signal analysis begins — signal work may not revisit this choice to chase performance.

## What this means for the algorithm

The selected estimator becomes the single hedge path for notebooks 05-12; its refit cadence and lookback become frozen or walk-forward-reoptimized parameters per walk_forward_config.yaml.